In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sktechno710/collabspace-events/collabspace_events.csv


In [21]:
# data loading
df = pd.read_csv('/kaggle/input/datasets/sktechno710/collabspace-events/collabspace_events.csv')

In [22]:
df

,user_id,event_date,event_type
0,1000,2025-01-06,login
1,1000,2025-01-09,send_message
2,1000,2025-01-21,login
3,1000,2025-01-24,upload_file
4,1000,2025-01-25,send_message
...,...,...,...
13781,1499,2025-06-11,send_message
13782,1499,2025-06-15,send_message
13783,1499,2025-06-15,login
13784,1499,2025-06-25,upload_file


In [23]:
# Converting to pandas datetime format
df['event_date'] = pd.to_datetime(df['event_date'])

In [24]:
# Finding number of unique user id
df['user_id'].unique()

array([1000, 1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010,
       1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021,
       1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032,
       1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043,
       1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1054,
       1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065,
       1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076,
       1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087,
       1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098,
       1099, 1100, 1101, 1102, 1103, 1104, 1105, 1106, 1107, 1108, 1109,
       1110, 1111, 1112, 1113, 1114, 1115, 1116, 1117, 1118, 1119, 1120,
       1121, 1122, 1123, 1124, 1125, 1126, 1127, 1128, 1129, 1130, 1131,
       1132, 1133, 1134, 1135, 1136, 1137, 1138, 1139, 1140, 1141, 1142,
       1143, 1144, 1145, 1146, 1147, 1148, 1149, 11

In [25]:
# Extracting the last event date of user id 1001
list(df[df['user_id']==1001].tail(2)['event_date'])[1]

Timestamp('2025-06-24 00:00:00')

In [26]:
# Extracting time diff of last two activities of a specific user id
list(df[df['user_id']==1001].tail(2)['event_date'])[1] - list(df[df['user_id']==1001].tail(2)['event_date'])[0]

Timedelta('5 days 00:00:00')

In [27]:
# Creating new dataframe
new_df = pd.DataFrame(columns=['user_id', 'days_since_last_event','last_event_date'])
for id in df['user_id'].unique():
    a = list(df[df['user_id']==id].tail(2)['event_date'])[1]
    b = list(df[df['user_id']==id].tail(2)['event_date'])[0]
    new_df.loc[len(new_df)]=[id,a-b,a]

In [28]:
new_df

,user_id,days_since_last_event,last_event_date
0,1000,6 days,2025-04-11
1,1001,5 days,2025-06-24
2,1002,8 days,2025-04-10
3,1003,6 days,2025-06-13
4,1004,5 days,2025-03-24
...,...,...,...
495,1495,15 days,2025-06-29
496,1496,3 days,2025-06-29
497,1497,1 days,2025-04-06
498,1498,1 days,2025-06-27


In [29]:
# No Null Value Present
print(new_df.isnull().sum())

user_id                  0
days_since_last_event    0
last_event_date          0
dtype: int64


In [30]:
# Statistically Churn_Threshold_Days is the 95th Percentile of the days since last event
Churn_Threshold_Days = new_df['days_since_last_event'].quantile(0.95)
Churn_Threshold_Days

Timedelta('17 days 00:00:00')

In [31]:
current_date = df['event_date'].max()
current_date

Timestamp('2025-06-29 00:00:00')

In [32]:
# Extracting which user is not active from more that Churn_Threshold_Days
new_df[current_date - new_df['last_event_date']>Churn_Threshold_Days]

,user_id,days_since_last_event,last_event_date
0,1000,6 days,2025-04-11
2,1002,8 days,2025-04-10
4,1004,5 days,2025-03-24
23,1023,15 days,2025-04-12
33,1033,31 days,2025-04-09
...,...,...,...
485,1485,12 days,2025-04-14
486,1486,4 days,2025-03-31
490,1490,12 days,2025-03-30
493,1493,2 days,2025-04-11


In [33]:
# Making N
new_df['IsChurned'] = np.where(current_date - new_df['last_event_date']>Churn_Threshold_Days, 1, 0)

In [34]:
new_df

,user_id,days_since_last_event,last_event_date,IsChurned
0,1000,6 days,2025-04-11,1
1,1001,5 days,2025-06-24,0
2,1002,8 days,2025-04-10,1
3,1003,6 days,2025-06-13,0
4,1004,5 days,2025-03-24,1
...,...,...,...,...
495,1495,15 days,2025-06-29,0
496,1496,3 days,2025-06-29,0
497,1497,1 days,2025-04-06,1
498,1498,1 days,2025-06-27,0


In [35]:
new_df.to_csv('aggregate_churn.csv',index=False)